# 4주차 ③ 로지스틱 회귀 — 실습 7-A · 7 · 7-B · 8

**목표**: 시그모이드와 BCE 가 **무엇을 하는지 직접 돌린 숫자로** 확인하고,
2교시·3교시와 **똑같은 학습 루프**로 이진 분류를 학습시킨 뒤, 모델이 그은 경계를 눈으로 본다.

> **실행 전 확인** — 우측 상단 커널 이름이 **`Python (dl2026)`** 인지 보세요.
> 아니면 커널명을 눌러 `Python (dl2026)` 을 선택하세요. (2주차 실습 8에서 다룬 그것입니다.)

### 이 노트북의 순서

| 셀 | 실습 | 내용 |
|:---:|---|---|
| 1 | **7-A** | 시그모이드 — 점수를 확률로 바꾸는 압축기를 찍어 본다 |
| 2~4 | **7** | 두 덩어리 데이터 → 학습 → 정확도 |
| 5~6 | **7-B** | 채점표 실험 — **MSE 로 바꾸면 왜 망하는가** ★★ |
| 부록 | (선택) | 시그모이드를 두 번 씌우면? |
| 7 | **8** | 결정 경계 색칠하기 |

> **오늘의 한 문장**: **학습 루프는 안 바뀐다. 바뀌는 것은 모델의 모양과 채점표뿐이다.**

---

## ★ 이 노트북에는 빈칸이 5곳 있습니다

| 빈칸 | 셀 | 채울 것 | 줄 수 |
|:---:|:---:|---|:---:|
| ① | 1 | 점수를 확률로 바꾸는 함수 | 1 |
| ② | 3 | **학습 루프 다섯 단계** ★ | 5 |
| ③ | 4 | 확률로 바꾸고 0.5 로 판정 | 2 |
| ④ | 6 | 두 채점표(BCE / MSE) | 2 |
| ⑤ | 7 | 격자의 모든 점에 대한 확률 | 1 |

> **빈칸 ② 가 오늘의 핵심입니다.** 실습 6(`04_linear_regression.ipynb` 셀 4)과
> **한 글자도 다르지 않습니다.** 보지 말고 먼저 써 보세요.

> 📎 **정답은 각 빈칸 셀 바로 아래 “▶ 정답 보기” 를 눌러 펼치면 나옵니다.**
> **먼저 스스로 써서 돌려 본 뒤에** 펼치세요. 펼치기 전에 아래 증상 표부터 보면 더 남습니다.

> ⚠️ **빈칸을 안 채워도 오류가 안 나는 곳이 있습니다.** (①③⑤)
> **오류가 없다고 맞는 것이 아닙니다.** 아래 표의 *"안 채웠을 때 나오는 것"* 과 비교하세요.

> **막히면 — 빈칸별 증상** (실제로 돌려서 확인한 것)
> | 증상 | 원인 |
> |---|---|
> | 확률이 `-6.0000`, `20.0000` 처럼 **0~1 을 벗어난다** | 빈칸 ① 미완성 — 시그모이드를 안 씌웠다 |
> | 손실이 `0.4487` **에서 300 epoch 내내 그대로** | 빈칸 ② 미완성 — 루프가 비어 있다 |
> | 손실이 줄었다 늘었다 한다 | 빈칸 ②에서 `optimizer.zero_grad()` 를 빼먹었다 (기울기 누적) |
> | `AttributeError: 'NoneType' object has no attribute 'backward'` | 빈칸 ② 또는 ④ 미완성 — `loss` 가 아직 `None` |
> | 정확도가 **0.0%** 로 나오고 "확률"이 음수다 | 빈칸 ③ 미완성 — 확률·판정을 안 만들었다 |
> | 셀 7 그림의 **검은 선이 엉뚱한 곳**에 있고 색 범위가 이상하다 | 빈칸 ⑤ 미완성 — 확률이 아니라 **점수**를 칠하고 있다 |
> | `ValueError: Target size ... must be the same as input size` | `y` 의 shape 이 `(200,)` 이다 → 셀 2 의 `assert` 가 먼저 잡아 준다 |
> | 정확도가 **50%** 로 고정된다 | 모델과 손실에 **시그모이드를 두 번** 씌웠다 (부록 셀 참고) |
> | 경계 그림이 완전히 뒤집혀 있다 | 셀 6 을 고쳐 쓰다 `model` 을 덮어썼다 → **셀 3 을 다시 실행** |

> **정답을 채우면** 정확도 **99.5%**, 최종 손실 **0.0278** 이 나옵니다 (시드 0 기준).

---

## 실습 7-A — 시그모이드 만져 보기

모델은 점마다 **"1 일 것 같은 정도"를 점수 하나**로 매깁니다. 점수는 `-9.62` 나 `+7.65` 처럼 **아무 숫자**입니다.
그 점수를 **0~1 사이로 눌러 담는 압축기**가 시그모이드입니다.

데이터도 모델도 아직 만들지 않습니다. **압축기만** 봅니다.

In [ ]:
# 셀 1 — 점수를 확률로 바꾸는 압축기   ★ 빈칸 ①
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"    # Mac 은 "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

for z in [-6, -2, -1, 0, 1, 2, 6, 20]:
    # ─────────────────────────────────────────────────────────
    # 빈칸 ①  점수를 0~1 확률로 바꾼다   (한 줄)
    #         힌트:  p = torch.________(torch.tensor(float(z)))
    # ─────────────────────────────────────────────────────────
    p = torch.tensor(float(z))          # ← 이 줄을 고치세요

    print(f"점수 {z:+3d}  →  확률 {p:.4f}")

zs = torch.linspace(-8, 8, 200)
plt.figure(figsize=(6, 3.5))
plt.plot(zs, torch.sigmoid(zs))
plt.axhline(0.5, color="gray", linestyle=":")
plt.axvline(0,   color="gray", linestyle=":")
plt.xlabel("점수 (모델이 낸 날것 출력)"); plt.ylabel("확률")
plt.title("시그모이드 — 점수를 0~1 로 눌러 담는다")
plt.show()

<details>
<summary><b>▶ 빈칸 ① 정답 보기</b> &nbsp;<sub>(먼저 스스로 써 본 뒤에 펼치세요)</sub></summary>

```python
p = torch.sigmoid(torch.tensor(float(z)))
```

**왜** — `torch.sigmoid` 가 **아무 숫자 하나를 0~1 사이로 눌러 담는** 압축기입니다.
안 씌우면 `-6`, `20` 이 그대로 찍혀 **확률이 될 수 없는 값**이 나옵니다.

</details>

```
점수  -6  →  확률 0.0025
점수  -2  →  확률 0.1192
점수  -1  →  확률 0.2689
점수  +0  →  확률 0.5000      ← 점수 0 이 딱 반반
점수  +1  →  확률 0.7311
점수  +2  →  확률 0.8808
점수  +6  →  확률 0.9975
점수 +20  →  확률 1.0000      ← 6 과 20 이 거의 같다
```

### 표에서 세 가지를 찾아보세요

| 찾을 것 | 어디에 쓰이나 |
|---|---|
| ① 확률이 **0.5** 가 되는 점수는? | 실습 8 — 결정 경계가 "점수 0 인 곳"인 이유 |
| ② 점수가 아무리 커져도 확률은? | **1 을 못 넘는다** → 확률로 쓸 수 있는 이유 |
| ③ 점수 **6 과 20** 의 확률 차이는? | **거의 없다** → 실습 7-B 에서 MSE 가 망가지는 이유 ★ |

> **③ 을 꼭 기억하세요.** 점수가 커지면 **확률이 더 이상 안 움직입니다.**
> 그래프의 **양 끝이 납작**하죠. 이 납작함이 조금 뒤에 사고를 칩니다.

> **추가 미션**: 확률을 **0.9** 로 만들려면 점수가 몇이어야 할까요? 숫자를 바꿔 가며 찾아보세요.
> (반대로 0.1 이 되는 점수도 찾아보면 **부호만 바뀐다**는 것을 알게 됩니다.)

---

## 실습 7 — 로지스틱 회귀 이진 분류

**모델 구조는 회귀와 같습니다.** `nn.Linear` 그대로입니다.
바뀌는 것은 **출력을 어떻게 해석하는가(시그모이드)** 와 **어떻게 틀린 정도를 재는가(BCE)** 뿐입니다.

In [ ]:
# 셀 2 — 두 부류 데이터
torch.manual_seed(0)
N = 100
c0 = torch.randn(N, 2) + torch.tensor([-2.0, -2.0])    # 부류 0
c1 = torch.randn(N, 2) + torch.tensor([ 2.0,  2.0])    # 부류 1

X = torch.cat([c0, c1])                                 # (200, 2)
y = torch.cat([torch.zeros(N, 1), torch.ones(N, 1)])    # (200, 1)

assert y.ndim == 2, "y 는 (N, 1) 이어야 한다 — (N,) 이면 사고가 난다"   # ★ 습관으로 만들 것
print("X.shape :", X.shape, "| y.shape :", y.shape)

plt.scatter(c0[:, 0], c0[:, 1], s=12, label="부류 0")
plt.scatter(c1[:, 0], c1[:, 1], s=12, label="부류 1")
plt.legend(); plt.title("이진 분류 데이터"); plt.show()

이 그림을 **건강검진 수치 두 개(혈당·체중) → 당뇨인가 아닌가**라고 생각하세요.
가로축·세로축이 검사 수치이고, 색이 정답입니다. 지금부터 **이 둘을 가르는 기준**을 모델이 찾습니다.

입력이 **2차원**이 되었으므로 모델은 `nn.Linear(2, 1)` — **입력 2개, 출력 1개(점수)** 입니다.
`(200,2) @ (2,1) → (200,1)` 을 스스로 검산해 보세요.

> **`assert y.ndim == 2` 를 왜 넣었나 ★ — 학기 내내 반복되는 1순위 사고입니다.**
>
> | 손실 | `y` 가 `(200,)` 일 때 |
> |---|---|
> | `BCEWithLogitsLoss` | `ValueError: Target size ... must be the same as input size` → **멈춘다** (그나마 다행) |
> | `MSELoss` | **오류 없음.** 경고만 띄우고 `(200, 200)` 으로 브로드캐스팅해 **4만 개**의 차이를 평균 낸다 → **조용히 엉뚱하게 학습** |
>
> `y.squeeze()` 를 쓰다가 이 실수를 하는 경우가 가장 많고, `MSELoss` 쪽은 **오류가 안 나서 끝까지 모릅니다.**
> 앞으로 모델이 바뀔 때마다 *"**모델 출력과 정답의 shape 이 같은가**"* 를 먼저 보세요.

In [ ]:
# 셀 3 — 학습   ★★ 빈칸 ② — 오늘의 핵심
torch.manual_seed(0)                          # ★ 이 셀만 다시 돌려도 같은 값이 나오게
model     = nn.Linear(2, 1)                   # ★ 구조는 회귀와 같다 (입력만 2개)
loss_fn   = nn.BCEWithLogitsLoss()            # ★ 손실만 바뀐다
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

history = []
for epoch in range(300):
    # ─────────────────────────────────────────────────────────
    # 빈칸 ②  학습 루프 다섯 단계   (다섯 줄)
    #         실습 6 (04_linear_regression.ipynb 셀 4) 과 한 글자도 다르지 않습니다.
    #         보지 말고 먼저 써 보세요.
    #
    #           ① 순전파 :  score = model(__)        ← 시그모이드 없이!
    #           ② 손실   :  loss  = loss_fn(__, __)
    #           ⑤ 지우기 :  optimizer.__________()
    #           ③ 역전파 :  loss.__________()
    #           ④ 갱신   :  optimizer.____()
    # ─────────────────────────────────────────────────────────
    score = model(X)
    loss  = loss_fn(score, y)



    history.append(loss.item())
    if epoch % 50 == 0:
        print(f"epoch {epoch:3d} | loss {loss.item():.4f}")
print(f"학습 끝 | loss {history[-1]:.4f}")

<details>
<summary><b>▶ 빈칸 ② 정답 보기</b> &nbsp;<sub>(오늘의 핵심 — 보지 말고 먼저 써 보세요)</sub></summary>

```python
    score = model(X)                          # ① 순전파 (시그모이드 없이!)
    loss  = loss_fn(score, y)                 # ② 손실

    optimizer.zero_grad()                     # ⑤ 지우기 (앞으로 옮겼다)
    loss.backward()                           # ③ 역전파
    optimizer.step()                          # ④ 갱신
```

**확인할 것 세 가지**

| 점검 | 이유 |
|---|---|
| `model(X)` 에 시그모이드를 **안 씌웠나** | `BCEWithLogitsLoss` 가 안에서 처리한다 (`WithLogits`) |
| `optimizer.zero_grad()` 가 `backward()` **앞**에 있나 | 빼먹으면 기울기가 누적돼 손실이 들쭉날쭉해진다 |
| `loss.backward()` → `optimizer.step()` **순서** | 기울기를 구한 뒤에 갱신한다 |

**이 다섯 줄은 실습 6(`04_linear_regression.ipynb` 셀 4)과 한 글자도 다르지 않습니다.**
직접 열어 놓고 줄 단위로 맞춰 보세요.

</details>

```
epoch   0 | loss 0.4487
epoch  50 | loss 0.0635
epoch 100 | loss 0.0441
epoch 150 | loss 0.0365
epoch 200 | loss 0.0324
epoch 250 | loss 0.0296
학습 끝 | loss 0.0278
```

실습 6 과 **줄 단위로 비교**해 보세요. 다른 것은 **두 곳뿐**인데, 이유가 서로 다릅니다.

```
nn.Linear(1, 1)  →  nn.Linear(2, 1)          입력 특성이 2개라서 (회귀여도 같다)
nn.MSELoss()     →  nn.BCEWithLogitsLoss()   분류라서  ★ 이 한 줄이 오늘의 핵심
```

**학습 루프는 한 글자도 안 바뀌었습니다.**

> **왜 `torch.manual_seed(0)` 를 또 쓰나 ★**: `nn.Linear` 는 만들어질 때 **무작위로 초기값을 뽑습니다.**
> 셀 2 를 거치지 않고 **이 셀만 다시 실행하면** 뽑히는 값이 달라져 위 숫자와 어긋납니다.
> 모델을 만들기 직전에 시드를 고정해 두면 몇 번을 돌리든 같은 자리에서 출발합니다.

> **함정 ★ — 여기서 시그모이드를 씌우지 마세요.**
> 이름의 **`WithLogits`** 가 *"점수(로짓)를 그대로 넣으라"* 는 뜻입니다.
> 시그모이드는 **이 함수가 안에서 처리**합니다.
> ```
> ✗  prob  = torch.sigmoid(model(X));  loss = nn.BCELoss()(prob, y)
> ○  score = model(X);                 loss = nn.BCEWithLogitsLoss()(score, y)
> ```
> 따로 계산하면 점수가 20 정도만 돼도 float32 에서 시그모이드가 정확히 `1.0` 이 되어 `log(0)` 이 생기고,
> `BCELoss` 는 이를 −100 에서 잘라 버려 **기울기가 0** 이 됩니다(그 데이터로는 학습이 멈춤).
> 합쳐 계산하면 그 문제가 없습니다 — **수치 안정성**. **중간고사 출제 지점입니다.**
> (두 번 씌우면 어떻게 되는지는 **부록 셀**에서 직접 돌려 봅니다.)

In [ ]:
# 셀 4 — 정확도 확인   ★ 빈칸 ③
with torch.no_grad():                          # ★ 평가에는 그래프가 필요 없다
    score = model(X)

    # ─────────────────────────────────────────────────────────
    # 빈칸 ③  점수 → 확률 → 판정   (두 줄)
    #         힌트:  prob = torch.________(score)      ← 여기서만 시그모이드
    #                pred = (prob __ 0.5).float()      ← 0.5 를 넘으면 1
    # ─────────────────────────────────────────────────────────
    prob = score                               # ← 이 두 줄을
    pred = score                               # ← 고치세요

    acc   = (pred == y).float().mean()

print(f"정확도 : {acc.item()*100:.1f}%")
print("\n점수 → 확률 → 예측 (앞 5개)")
for i in [0, 1, 100, 101, 102]:
    print(f"  점수 {score[i].item():7.2f} → 확률 {prob[i].item():.3f} "
          f"→ 예측 {int(pred[i].item())} | 정답 {int(y[i].item())}")

<details>
<summary><b>▶ 빈칸 ③ 정답 보기</b></summary>

```python
    prob = torch.sigmoid(score)                # ★ 여기서만 시그모이드를 씌운다
    pred = (prob > 0.5).float()                # ★ 0.5 를 넘으면 1
```

**왜 여기서만** — 학습 중에는 `BCEWithLogitsLoss` 가 안에서 씌워 줬습니다.
**사람이 확률을 읽고 판정하려는 지금**만 직접 씌웁니다.

**`.float()` 은 왜** — `(prob > 0.5)` 는 `True/False` 입니다.
`y` 가 `0.0/1.0` 이므로 같은 자료형으로 맞춰야 `pred == y` 비교가 제대로 됩니다.

</details>

```
정확도 : 99.5%

점수 → 확률 → 예측 (앞 5개)
  점수   -9.62 → 확률 0.000 → 예측 0 | 정답 0
  점수   -7.20 → 확률 0.001 → 예측 0 | 정답 0
  점수    6.60 → 확률 0.999 → 예측 1 | 정답 1
  점수    7.65 → 확률 1.000 → 예측 1 | 정답 1
  점수    0.91 → 확률 0.712 → 예측 1 | 정답 1     ← 확신이 약하다
```

**한 줄에 세 마디가 다 보입니다** — `점수` → `확률` → `예측`.
셀 1 에서 찍어 본 그 압축기가 **여기서 그대로 동작**하고 있습니다.

> **점수의 크기 = 경계에서의 거리 ★**
> 점수 `-9.62` 나 `+7.65` 인 데이터는 경계선에서 **멀리 떨어진 안전지대**에 있습니다 (확률이 0.000·1.000 에 딱 붙음).
> 반면 마지막 줄의 `0.91` 은 **경계선 바로 옆에서 아슬아슬하게 판정된 점**입니다.
> **부호는 어느 쪽인지, 크기는 얼마나 확신하는지**를 말해 줍니다. 실습 8 에서 이 점의 위치를 눈으로 확인하세요.

> `torch.sigmoid` 를 **여기서 처음** 썼습니다. 학습 중에는 `BCEWithLogitsLoss` 가 안에서 처리했으므로 필요 없었고,
> **사람이 확률을 읽고 싶을 때만** 씌웁니다.

> 정확도가 100% 에 가까운 것은 **데이터를 그렇게 만들었기 때문**입니다.
> **학습에 쓴 데이터로 평가**하기도 했고요. 6주차에 **정확도만 보면 안 되는 이유**를 배웁니다.

---

## 실습 7-B — 채점표 실험: MSE vs BCE ★★

**"왜 MSE 가 아니라 BCE 인가"** 를 설명 대신 **직접 돌려서** 확인합니다.

**상황**: 정답이 **1** 인 데이터 하나. 모델이 "1 일 확률"을 얼마로 줬느냐에 따라 두 채점표가 어떻게 다른가.

In [ ]:
# 셀 5 — 두 채점표 비교 (벌점과 "고치는 힘")
import torch.nn.functional as F

print(f"{'확률':>6} | {'BCE 벌점':>9} {'MSE 벌점':>9} | {'BCE 고치는 힘':>14} {'MSE 고치는 힘':>14}")
for p in [0.9, 0.5, 0.1, 0.01, 0.001]:
    p_t    = torch.tensor(p)
    score0 = torch.log(p_t / (1 - p_t))       # 이 확률이 나오는 점수
    y_t    = torch.tensor(1.0)                # 정답은 1

    s1  = score0.clone().requires_grad_(True)
    bce = F.binary_cross_entropy_with_logits(s1, y_t);  bce.backward()

    s2  = score0.clone().requires_grad_(True)
    mse = (torch.sigmoid(s2) - y_t) ** 2;               mse.backward()

    print(f"{p:>6} | {bce.item():>9.3f} {mse.item():>9.3f} | "
          f"{abs(s1.grad.item()):>14.4f} {abs(s2.grad.item()):>14.5f}")

```
    확률 |    BCE 벌점    MSE 벌점 |      BCE 고치는 힘      MSE 고치는 힘
   0.9 |     0.105     0.010 |         0.1000        0.01800
   0.5 |     0.693     0.250 |         0.5000        0.25000
   0.1 |     2.303     0.810 |         0.9000        0.16200
  0.01 |     4.605     0.980 |         0.9900        0.01960
 0.001 |     6.908     0.998 |         0.9990        0.00200
```

| 확률 | 0.9 | 0.5 | 0.1 | 0.01 | 0.001 |
|---|---|---|---|---|---|
| 상황 | 거의 맞힘 | 모르겠음 | 틀림 | 확신하고 틀림 | **아주 확신하고 틀림** |
| **BCE 고치는 힘** | 0.10 | 0.50 | 0.90 | 0.99 | **1.00 — 계속 커진다** ○ |
| **MSE 고치는 힘** | 0.018 | 0.25 | 0.16 | 0.020 | **0.002 — 도로 작아진다** ✗ |

**가장 크게 틀린 마지막 칸을 보세요.** 더 크게 고쳐야 할까요, 덜 고쳐야 할까요?
**MSE 는 크게 틀릴수록 오히려 덜 고칩니다** (`0.16 → 0.020 → 0.002`).
셀 1 에서 본 **양 끝의 납작함** 때문입니다 — 점수가 멀리 가면 확률이 안 움직이니 고칠 방향도 안 잡힙니다.

> **"틀린 만큼 그대로" 를 직접 검산해 보세요 ★ — 공식이 필요 없습니다.**
> 정답은 **1** 인데 모델이 **0.001** 이라고 답했습니다. **틀린 정도는 `1 - 0.001 = 0.999`.**
> 표의 BCE 칸을 보세요 — **0.9990.** 똑같습니다.
> 확률 0.9 면 틀린 정도 0.1, 힘도 **0.1000**. 확률 0.5 면 0.5, 힘도 **0.5000**.
> **`1 - 확률` 열과 `BCE 고치는 힘` 열이 전부 같습니다.** BCE 는 **틀린 양을 그대로** 밀어 줍니다.

> **MSE 는 확률 0.9 에서도 0.018 로 작은데요?** — 맞습니다.
> MSE 의 고치는 힘은 **양쪽 끝에서 모두** 작아집니다. 그런데 **오른쪽 끝(거의 맞힘)이 작은 건 문제가 아닙니다.**
> 이미 맞았으니 조금만 고치는 게 맞죠. 문제는 **왼쪽 끝(크게 틀림)에서도 작아진다**는 것입니다.

### 이제 진짜로 돌려 봅니다 — 손실만 바꿉니다 ★★

**같은 데이터, 같은 모델, 같은 루프, 같은 epoch. 딱 한 줄만 다릅니다.**
단, **"확신하고 틀린" 자리에서 출발**시킵니다 (가중치를 일부러 반대 방향으로 크게 잡음).

In [ ]:
# 셀 6 — 손실만 바꿔 같은 학습을 돌린다   ★ 빈칸 ④
def train_with(loss_kind, epochs=300):
    torch.manual_seed(0)
    m = nn.Linear(2, 1)
    with torch.no_grad():                      # ★ 일부러 "확신하고 틀린" 곳에서 출발
        m.weight.copy_(torch.tensor([[-4.0, -4.0]]))
        m.bias.zero_()

    opt  = torch.optim.SGD(m.parameters(), lr=0.1)
    hist, g0 = [], None
    for e in range(epochs):
        score = m(X)
        # ─────────────────────────────────────────────────────
        # 빈칸 ④  두 채점표   (각 한 줄)
        #         힌트:  loss = nn.__________________()(score, y)
        #                loss = nn.MSELoss()(torch.________(score), y)
        #         ※ MSE 쪽만 시그모이드를 직접 씌웁니다. 왜일까요?
        # ─────────────────────────────────────────────────────
        if loss_kind == "BCE":
            loss = None                        # ← 고치세요
        else:
            loss = None                        # ← 고치세요

        opt.zero_grad(); loss.backward()
        if e == 0:
            g0 = m.weight.grad.abs().max().item()               # 첫 걸음의 기울기
        opt.step(); hist.append(loss.item())

    with torch.no_grad():
        acc = ((torch.sigmoid(m(X)) > 0.5).float() == y).float().mean()
    return hist, acc.item(), g0

for kind in ["BCE", "MSE"]:
    hist, acc, g0 = train_with(kind)
    print(f"{kind} | 손실 {hist[0]:7.4f} → {hist[-1]:.4f} | 정확도 {acc*100:5.1f}% "
          f"| 첫 걸음의 기울기 {g0:.5f}")

<details>
<summary><b>▶ 빈칸 ④ 정답 보기</b></summary>

```python
        if loss_kind == "BCE":
            loss = nn.BCEWithLogitsLoss()(score, y)             # ← 이 줄만
        else:
            loss = nn.MSELoss()(torch.sigmoid(score), y)        # ← 다르다
```

**※ MSE 쪽만 시그모이드를 직접 씌운 이유** — `BCEWithLogitsLoss` 는 **안에서 씌워 주지만**,
`MSELoss` 는 그냥 두 숫자의 차이를 제곱할 뿐입니다.
**확률끼리 비교하려면** `torch.sigmoid` 를 직접 붙여 줘야 두 채점표가 **같은 조건**에서 겨룹니다.
(안 붙이면 점수(로짓)와 0/1 을 비교하게 되어 비교 자체가 성립하지 않습니다.)

</details>

```
BCE | 손실 15.9442 → 0.0274 | 정확도  99.5% | 첫 걸음의 기울기 2.01211
MSE | 손실  0.9883 → 0.9878 | 정확도   0.5% | 첫 걸음의 기울기 0.00113
```

- **BCE**: 손실 **15.94** 에서 출발했습니다. *"엄청 틀렸다"* 를 제대로 알아채고 **0.027 까지 내려가 99.5%** 로 회복했습니다.
- **MSE**: 손실이 **0.9883 → 0.9878.** 300 epoch 동안 **거의 안 움직였습니다.** 정확도 **0.5%** — 전부 거꾸로 맞혔습니다.
- 이유는 맨 오른쪽 칸 하나로 끝납니다 — **첫 걸음의 기울기가 2.01 대 0.00113, 약 1,800배 차이.**

> **MSE 는 틀린 걸 모르는 게 아닙니다.** 손실 0.99 면 최악에 가깝다는 걸 압니다.
> 그런데 **어느 쪽으로 가야 할지의 신호가 0.001 밖에 안 됩니다.** 알면서도 못 고치는 것입니다.
> 셀 5 표의 마지막 칸이 실제로 일어난 모습입니다.

> **그럼 MSE 는 분류에 절대 못 쓰나요?** — 운 좋은 자리에서 출발하면 MSE 도 됩니다
> (이 데이터에서 기본 초기값으로 출발시키면 99.0% 까지 갑니다).
> **문제는 한 번 확신하고 틀리면 빠져나올 수 없다**는 것입니다. 위 MSE 를 **3000 epoch** 로 늘려도 손실 0.978, 정확도 1.5% 입니다.
> 층이 깊어질수록 이런 자리에 빠질 일이 많아지니, **분류에는 처음부터 BCE** 를 씁니다.

### 부록 셀 (선택) — 시그모이드를 두 번 씌우면?

빨리 끝낸 사람만 돌려 보세요. **하면 안 되는 것을 일부러** 해 봅니다.
(이 셀은 `model` 을 건드리지 않으므로 건너뛰어도 뒤 셀에 영향이 없습니다.)

In [ ]:
# 부록 셀 (선택) — 하면 안 되는 것을 일부러 해 본다
torch.manual_seed(0)
bad = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid())     # ✗ 모델 안에 시그모이드
opt = torch.optim.SGD(bad.parameters(), lr=0.1)
for e in range(300):
    loss = nn.BCEWithLogitsLoss()(bad(X), y)           # ✗ 여기서 또 한 번
    opt.zero_grad(); loss.backward(); opt.step()

with torch.no_grad():
    pb = torch.sigmoid(bad(X))
    print(f"손실 {loss.item():.4f} | 정확도 {((pb>0.5).float()==y).float().mean()*100:.1f}% "
          f"| 확률 범위 {pb.min():.3f} ~ {pb.max():.3f}")

```
손실 0.5176 | 정확도 50.0% | 확률 범위 0.500 ~ 0.731
```

시그모이드 출력(0~1)을 **다시 점수로 취급**하니 최종 확률이 **0.5 ~ 0.73 사이에 갇힙니다.**
확률이 항상 0.5 를 넘으니 **전부 1 로 예측 → 정확도 50%**, 손실도 원리적으로 **0.5 아래로 내려갈 수 없습니다.**

**오류가 하나도 안 납니다.** 그래서 더 위험합니다 — `BCEWithLogitsLoss` 를 쓸 때는 **모델 마지막에 시그모이드를 넣지 마세요.**

---

## 실습 8 — 결정 경계 색칠하기

모델이 그은 경계를 봅니다. 식을 풀지 않습니다 — **화면의 모든 점(4만 개)에 모델을 돌려 확률로 색칠**하고,
확률이 **0.5 인 곳**을 컴퓨터가 잇게 합니다.

> **실행 전 확인**: 이 셀은 **셀 3 에서 정상 학습된 `model`** 을 씁니다.
> 셀 6 에서 망가뜨린 것은 **함수 안의 `m`** 이라 `model` 은 멀쩡합니다.
> 그림이 이상하면 **셀 3 을 다시 실행**하세요.

> **코드를 돌리기 전에 공책에 먼저 그어 보세요 ★**
> 두 덩어리 중심이 (−2, −2) 와 (2, 2) 입니다. 경계선이 **어디를 어떻게** 지날까요?

In [ ]:
# 셀 7 — 모델이 본 세상을 색으로 칠한다   ★ 빈칸 ⑤
xx, yy = torch.meshgrid(torch.linspace(-6, 6, 200),
                        torch.linspace(-6, 6, 200), indexing="xy")
grid = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)     # (40000, 2) ← 화면의 모든 점

with torch.no_grad():
    # ─────────────────────────────────────────────────────────
    # 빈칸 ⑤  격자의 모든 점에 대한 확률   (한 줄)
    #         힌트:  torch.________(model(grid)).reshape(xx.shape)
    #         ※ 셀 4 에서 한 것과 똑같습니다 — 점수를 확률로.
    # ─────────────────────────────────────────────────────────
    prob_grid = model(grid).reshape(xx.shape)      # ← 이 줄을 고치세요

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, prob_grid, levels=20, cmap="coolwarm", alpha=0.7)
plt.colorbar(label="부류 1 일 확률")
plt.contour(xx, yy, prob_grid, levels=[0.5], colors="black", linewidths=2)   # 확률 0.5 인 곳
plt.scatter(c0[:, 0], c0[:, 1], s=12, label="부류 0")
plt.scatter(c1[:, 0], c1[:, 1], s=12, label="부류 1")
plt.legend(); plt.title("모델이 본 세상 — 검은 선이 확률 0.5 (결정 경계)")
plt.show()

<details>
<summary><b>▶ 빈칸 ⑤ 정답 보기</b></summary>

```python
    prob_grid = torch.sigmoid(model(grid)).reshape(xx.shape)    # 점마다 확률을 구한다
```

**왜** — 아래 `plt.contour(..., levels=[0.5], ...)` 가 **확률 0.5 인 곳**에 검은 선을 긋습니다.
시그모이드를 빼면 칠해지는 것이 **점수**라서 `0.5` 는 아무 의미 없는 자리가 되고,
검은 선이 엉뚱한 곳에 그어집니다. (점수 기준이라면 경계는 **0** 입니다.)

**셀 4 에서 한 것과 똑같습니다** — 점수를 확률로. 다만 대상이 데이터 200개가 아니라 **화면의 4만 개 점**입니다.

</details>

### 그림에서 세 가지를 읽어 보세요

| 읽을 것 | 보이는 것 | 연결 |
|---|---|---|
| ① 검은 선의 위치 | 두 덩어리 **사이를 대각선으로** 지나간다 | 모델이 학습한 전부 |
| ② 선에서 멀어질수록 | 색이 **진해진다** = 확률이 0 이나 1 에 가까워진다 | 셀 1 의 시그모이드 그래프 |
| ③ **선 근처의 옅은 띠** | 여기 있는 점이 **확신이 약한 점** | 셀 4 의 `점수 0.91 → 확률 0.712` 가 이 띠 안에 있다 ★ |

> **여기가 5주차로 넘어가는 다리입니다.**
> 이 그림에서 경계는 **직선**입니다. `nn.Linear` 하나는 **직선 하나**밖에 못 긋습니다.
> 두 덩어리가 섞여 있거나 **원 모양**으로 배치되어 있으면 직선으로는 절대 못 가릅니다.
> 그래서 **층을 쌓고 그 사이에 활성함수**를 넣습니다 — 그게 다음 주차 신경망(MLP)입니다.
>
> 오늘 쓴 **색칠 방법도 5주차에서 그대로** 씁니다 (`06_xor_and_activations` 에서 세 모델의 경계를 나란히 칠합니다).
> 다만 그때는 **경계가 직선이 아닙니다.**

> **왜 굳이 색칠하나요? 선 하나 그으면 되는데** — `w1·x1 + w2·x2 + b = 0` 을 풀면 직선 식이 나오긴 합니다.
> 하지만 **층을 쌓으면 식으로 못 풉니다.** 색칠하는 방법은 **어떤 모델에도 통합니다.**

---

### 이 노트북 체크리스트

- [ ] **점수 0 = 확률 0.5** 를 직접 찍어서 확인했다 (셀 1)
- [ ] `nn.Linear(2, 1)` 인 이유를 shape 으로 설명할 수 있다 (셀 2)
- [ ] 학습 루프가 실습 6 과 **똑같다**는 것을 **보지 않고 써서** 확인했다 ★ (셀 3)
- [ ] 회귀와 분류의 코드 차이가 **손실함수 한 줄**이라는 것을 안다
- [ ] `BCEWithLogitsLoss` 에 시그모이드를 따로 붙이지 않는 이유를 안다 ★
- [ ] 정확도가 90% 이상 나왔다 (셀 4)
- [ ] **손실만 바꿔 99.5% vs 0.5%** 를 직접 봤다 ★★ (셀 6)
- [ ] "MSE 는 크게 틀릴수록 덜 고친다"를 **표의 숫자로** 말할 수 있다 (셀 5)
- [ ] 결정 경계를 색칠해 보고 **직선 하나의 한계**를 말할 수 있다 (셀 7)

### 과제 제출 전 확인

- [ ] 셀 1~7 이 모두 실행되고 **출력이 저장된 상태**다 (`Ctrl+S`)
- [ ] 회고에 *"실습 7-B 에서 MSE 가 0.5% 에 머문 이유"* 를 자기 말로 적었다 ★